INITIALIZE & TRAIN:

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Import from organized wind_farm_control package
from wind_farm_control.environments import FlorisMultiAgentEnv
from wind_farm_control.models import Actor, Critic

import numpy as np
import supersuit as ss
from skrl.envs.wrappers.torch import wrap_env
from skrl.memories.torch import RandomMemory
from skrl.multi_agents.torch.mappo import MAPPO, MAPPO_DEFAULT_CONFIG
from skrl.trainers.torch import ParallelTrainer


class SuperSuitParallelAdapter:
    def __init__(self, vec_env, template_env):
        self.vec_env = vec_env
        self.unwrapped = self
        self.possible_agents = list(template_env.possible_agents)
        self.agents = list(template_env.possible_agents)
        self.num_agents = len(self.possible_agents)
        self.num_envs = vec_env.num_envs // self.num_agents

        self.observation_spaces = template_env.observation_spaces
        self.action_spaces = template_env.action_spaces
        self.state_space = template_env.state_space
        self.shared_observation_spaces = template_env.shared_observation_spaces
        self.render_mode = getattr(template_env, "render_mode", None)

    def _to_dict(self, flat_values):
        return {
            agent: np.asarray(flat_values[index :: self.num_agents])
            for index, agent in enumerate(self.possible_agents)
        }

    def _shared_state_from_obs(self, observations):
        per_agent = [observations[agent] for agent in self.possible_agents]
        return np.concatenate(per_agent, axis=1)

    def reset(self, seed=None, options=None):
        outputs = self.vec_env.reset(seed=seed, options=options)
        if isinstance(outputs, tuple):
            observations = outputs[0]
        else:
            observations = outputs

        observation_dict = self._to_dict(observations)
        shared_state = self._shared_state_from_obs(observation_dict)
        infos = {agent: {"state": shared_state} for agent in self.possible_agents}
        return observation_dict, infos

    def step(self, actions):
        sample_agent = self.possible_agents[0]
        action_shape = self.action_spaces[sample_agent].shape
        flat_actions = np.zeros(
            (self.vec_env.num_envs,) + action_shape, dtype=np.float32
        )

        for index, agent in enumerate(self.possible_agents):
            agent_actions = np.asarray(actions[agent], dtype=np.float32)
            flat_actions[index :: self.num_agents] = agent_actions

        observations, rewards, terminated, truncated, _ = self.vec_env.step(
            flat_actions
        )

        observation_dict = self._to_dict(observations)
        reward_dict = self._to_dict(rewards)
        terminated_dict = self._to_dict(terminated)
        truncated_dict = self._to_dict(truncated)

        shared_state = self._shared_state_from_obs(observation_dict)
        infos = {agent: {"state": shared_state} for agent in self.possible_agents}

        return observation_dict, reward_dict, terminated_dict, truncated_dict, infos

    def state(self):
        observations, _ = self.reset()
        return self._shared_state_from_obs(observations)

    def render(self, *args, **kwargs):
        return self.vec_env.render(*args, **kwargs)

    def close(self):
        self.vec_env.close()


# Env Setup + SuperSuit vectorization (for training)
template_env = FlorisMultiAgentEnv("../data_generation/farm_types/gch.yaml")
possible_agents = template_env.possible_agents
observation_spaces = template_env.observation_spaces
action_spaces = template_env.action_spaces
shared_observation_spaces = template_env.shared_observation_spaces

vec_env = ss.pettingzoo_env_to_vec_env_v1(template_env)
vec_env = ss.concat_vec_envs_v1(vec_env, 4, num_cpus=0, base_class="gymnasium")
adapted_env = SuperSuitParallelAdapter(vec_env, template_env)
train_env = wrap_env(adapted_env, wrapper="pettingzoo")

# memory object
memory = RandomMemory(
    memory_size=2000, num_envs=train_env.num_envs, device=train_env.device
)

# wrapped in a dictionary for MAPPO
memories = {agent_name: memory for agent_name in possible_agents}

# Model Sharing
shared_policy = Actor(
    observation_spaces["turbine_0"], action_spaces["turbine_0"], train_env.device
)

# Ensure the Critic uses the SHARED space (the 12-element one)
shared_value = Critic(
    shared_observation_spaces["turbine_0"],
    action_spaces["turbine_0"],
    train_env.device,
)

models = {a: {"policy": shared_policy, "value": shared_value} for a in possible_agents}

cfg_agent = MAPPO_DEFAULT_CONFIG.copy()
cfg_agent["random_timesteps"] = 0  # Start learning immediately
cfg_agent["learning_rate"] = 5e-4
cfg_agent["state_preprocessor"] = None  # Optional: helps with stability

agent = MAPPO(
    possible_agents=possible_agents,
    models=models,
    memories=memories,
    cfg=cfg_agent,
    observation_spaces=observation_spaces,
    action_spaces=action_spaces,
    device=train_env.device,
    shared_observation_spaces=shared_observation_spaces,
)

# TRAIN on vectorized env
trainer = ParallelTrainer(
    env=train_env,
    agents=agent,
    cfg={"timesteps": 5000, "headless": True, "disable_progressbar": False},
)
trainer.train()

# Non-vectorized env for evaluation cells below
env = wrap_env(
    FlorisMultiAgentEnv("../data_generation/farm_types/gch.yaml"), wrapper="pettingzoo"
)

/home/nandan/SourceCode/wind-farm-control/.venv/lib/python3.13/site-packages/pettingzoo/utils/env.py:358: UserWarning: Your environment should override the observation_space function. Attempting to use the observation_spaces dict attribute.
  warnings.warn(
/home/nandan/SourceCode/wind-farm-control/.venv/lib/python3.13/site-packages/pettingzoo/utils/env.py:370: UserWarning: Your environment should override the action_space function. Attempting to use the action_spaces dict attribute.
  warnings.warn(
[skrl:INFO] Environment wrapper: Petting Zoo


AttributeError: Wrapped environment (ConcatVecEnv) does not have attribute 'num_agents'

EVALUATE

In [ ]:
# Import evaluation utilities
from wind_farm_control.evaluation import (
    evaluate_mappo_vs_baseline,
    plot_mappo_performance,
)

# Execute evaluation
df_mappo_results = evaluate_mappo_vs_baseline(agent, env, n_episodes=20)
print(df_mappo_results["Gain_%"].describe())
plot_mappo_performance(df_mappo_results)

HYPERPARAMETER TUNING

In [ ]:
# Import hyperparameter tuning utilities
from wind_farm_control.utils import run_hyperparameter_tuning

# Run hyperparameter tuning
study = run_hyperparameter_tuning(
    config_path="../data_generation/farm_types/gch.yaml",
    n_trials=10,
    training_timesteps=5000,
    eval_steps=100,
)